# Open-Set Recognition with MNIST: Reducing Network Agnostophobia

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ludwig-ai/ludwig/blob/main/examples/open_set_recognition/open_set_mnist.ipynb)

Standard image classifiers assign high-confidence predictions to *every* input — including images
from classes never seen during training. This is called **network agnostophobia**: the model is
incapable of saying "I don't know."

This notebook demonstrates three Ludwig models trained on MNIST digits **0–7** (known classes)
with digits **8–9** acting as the unknown/background:

| Model | Loss | Expected behaviour |
|-------|------|--------------------|
| CE Baseline | `softmax_cross_entropy` | High confidence even on unknown digits |
| Entropic Open-Set | `entropic_open_set` | Pushes unknown confidence toward uniform |
| Objectosphere | `objectosphere` | Creates a clear logit-norm gap between known and unknown |

**Paper:** Dhamija, Günther, Boult — *Reducing Network Agnostophobia*, NeurIPS 2018.
https://arxiv.org/abs/1811.04110

In [ ]:
!pip install ludwig torchvision --quiet

## Setup and data preparation

We download MNIST via `torchvision`, save each digit image as a PNG file, and build two CSVs:

- **`train.csv`** — digits 0–7 with their true labels, plus digits 8–9 labelled as `"background"`
- **`test.csv`** — digits 0–7 (known) and digits 8–9 (unknown) with their true labels

The training set teaches the open-set models what "background" looks like. The test set lets us
measure how confidently each model handles the unseen classes.

In [ ]:
import os
import csv
from pathlib import Path

import torch
from torchvision import datasets, transforms
from PIL import Image

# ── configuration ──────────────────────────────────────────────────────────────
DATA_DIR = Path("mnist_data")       # raw MNIST download
IMG_DIR  = Path("mnist_images")     # saved PNG files
KNOWN_CLASSES   = list(range(8))    # digits 0-7
UNKNOWN_CLASSES = [8, 9]            # background / unknown

# Limit samples per class so training is fast on CPU
MAX_TRAIN_KNOWN   = 500   # per known class
MAX_TRAIN_UNKNOWN = 500   # per unknown class (background)
MAX_TEST_KNOWN    = 200   # per known class
MAX_TEST_UNKNOWN  = 200   # per unknown class

IMG_DIR.mkdir(parents=True, exist_ok=True)

# ── download ───────────────────────────────────────────────────────────────────
mnist_train = datasets.MNIST(str(DATA_DIR), train=True,  download=True,
                              transform=transforms.ToTensor())
mnist_test  = datasets.MNIST(str(DATA_DIR), train=False, download=True,
                              transform=transforms.ToTensor())

print(f"Downloaded: {len(mnist_train)} train / {len(mnist_test)} test samples")

# ── helper: save image and return path ────────────────────────────────────────
def save_image(tensor: torch.Tensor, split: str, digit: int, idx: int) -> str:
    """Save a (1, H, W) float tensor as a grayscale PNG; return the file path."""
    folder = IMG_DIR / split / str(digit)
    folder.mkdir(parents=True, exist_ok=True)
    fpath = folder / f"{idx:05d}.png"
    img = Image.fromarray((tensor.squeeze(0).numpy() * 255).astype("uint8"), mode="L")
    img.save(fpath)
    return str(fpath)

# ── build train.csv ───────────────────────────────────────────────────────────
# class counter for capping per-class samples
from collections import defaultdict

def build_csv(dataset, csv_path: str, split: str,
              known_classes, unknown_classes,
              max_known: int, max_unknown: int,
              label_unknown_as_background: bool):
    """
    Walk *dataset*, save PNGs, write *csv_path*.

    label_unknown_as_background=True  → training split (unknown → "background")
    label_unknown_as_background=False → test split (unknown keeps true digit string)
    """
    counts_known   = defaultdict(int)
    counts_unknown = defaultdict(int)
    rows = []

    for global_idx, (img_tensor, digit) in enumerate(dataset):
        digit = int(digit)
        if digit in known_classes:
            if counts_known[digit] >= max_known:
                continue
            path  = save_image(img_tensor, split, digit, global_idx)
            label = str(digit)
            counts_known[digit] += 1
        elif digit in unknown_classes:
            if counts_unknown[digit] >= max_unknown:
                continue
            path  = save_image(img_tensor, split, digit, global_idx)
            label = "background" if label_unknown_as_background else str(digit)
            counts_unknown[digit] += 1
        else:
            continue
        rows.append({"image_path": path, "label": label})

        # stop early once all caps are met
        if (all(counts_known[c] >= max_known   for c in known_classes) and
            all(counts_unknown[c] >= max_unknown for c in unknown_classes)):
            break

    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["image_path", "label"])
        writer.writeheader()
        writer.writerows(rows)

    known_total   = sum(counts_known.values())
    unknown_total = sum(counts_unknown.values())
    print(f"  {csv_path}: {known_total} known, {unknown_total} unknown/background")
    return rows

print("Building train.csv ...")
train_rows = build_csv(
    mnist_train, "train.csv", "train",
    KNOWN_CLASSES, UNKNOWN_CLASSES,
    MAX_TRAIN_KNOWN, MAX_TRAIN_UNKNOWN,
    label_unknown_as_background=True,
)

print("Building test.csv ...")
test_rows = build_csv(
    mnist_test, "test.csv", "test",
    KNOWN_CLASSES, UNKNOWN_CLASSES,
    MAX_TEST_KNOWN, MAX_TEST_UNKNOWN,
    label_unknown_as_background=False,
)

print("Done.")

Let's quickly inspect what our CSVs look like.

In [ ]:
import pandas as pd

train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")

print("train.csv label distribution:")
print(train_df["label"].value_counts().sort_index())
print()
print("test.csv label distribution:")
print(test_df["label"].value_counts().sort_index())
print()
print(train_df.head())

## Discover background class index

Ludwig assigns integer indices to category labels based on frequency in the training data
(most frequent first, with index 0 reserved for `<UNK>`).  Before using `entropic_open_set`
or `objectosphere` loss we need to know the integer index that Ludwig assigns to the
`"background"` label.

The safest approach:
1. Run a short training job with the baseline config (standard cross-entropy).
2. Open `training_set_metadata.json` in the saved model directory.
3. Look up `output_features -> label -> str2idx -> "background"`.

We do exactly this below — first training the baseline, then extracting the index programmatically.

## Baseline: softmax cross-entropy

The baseline model is a standard image classifier trained with softmax cross-entropy.  It is
trained **only on known classes** (digits 0–7); digits 8–9 are excluded from its training data.

At test time the baseline will happily assign high confidence to unknown digits because it has
no mechanism to express uncertainty.

In [ ]:
import yaml

ENCODER = {
    "type": "stacked_cnn",
    "conv_layers": [
        {"num_filters": 32, "filter_size": 3, "pool_size": 2, "pool_stride": 2},
        {"num_filters": 64, "filter_size": 3, "pool_size": 2, "pool_stride": 2},
    ],
    "fc_layers": [{"output_size": 128, "dropout": 0.3}],
}

config_baseline = {
    "model_type": "ecd",
    "input_features": [{"name": "image_path", "type": "image", "encoder": ENCODER}],
    "output_features": [
        {"name": "label", "type": "category",
         "loss": {"type": "softmax_cross_entropy"}}
    ],
    "trainer": {"epochs": 10, "learning_rate": 0.001, "batch_size": 128},
}

print(yaml.dump(config_baseline, default_flow_style=False))

In [ ]:
from ludwig.api import LudwigModel

# The baseline is trained only on known classes — filter out the background rows
train_known_df = train_df[train_df["label"] != "background"].copy()
print(f"Training baseline on {len(train_known_df)} samples (known classes only)")

model_baseline = LudwigModel(config=config_baseline, logging_level="WARNING")
_, _, output_dir_baseline = model_baseline.train(
    dataset=train_known_df,
    experiment_name="open_set_mnist",
    model_name="baseline",
    skip_save_processed_input=True,
)
print(f"Baseline model saved to: {output_dir_baseline}")

### Find the background class index from `training_set_metadata.json`

Now that we have a trained model we can inspect its vocabulary.  The agnostophobia models will
be trained on the *full* training set (known + background), so the background label will appear
in the vocabulary.  We run a quick vocabulary fit to discover its index before training those
models.

In [ ]:
import json
from pathlib import Path

# The agnostophobia models are trained on the full dataset (including background).
# We need to know what index Ludwig will assign to "background" in *that* vocabulary.
# The simplest way is to train the entropic model first (or do a preprocessing run),
# but we can also use Ludwig's preprocessing API directly.

from ludwig.api import LudwigModel

# Build a minimal config for preprocessing only
config_for_vocab = {
    "model_type": "ecd",
    "input_features": [{"name": "image_path", "type": "image", "encoder": ENCODER}],
    "output_features": [
        {"name": "label", "type": "category",
         "loss": {"type": "softmax_cross_entropy"}}
    ],
    "trainer": {"epochs": 1, "batch_size": 128},
}

vocab_model = LudwigModel(config=config_for_vocab, logging_level="WARNING")
_, _, output_dir_vocab = vocab_model.train(
    dataset=train_df,  # full training set including background
    experiment_name="open_set_mnist",
    model_name="vocab_probe",
    skip_save_processed_input=True,
)

metadata_path = Path(output_dir_vocab) / "model" / "training_set_metadata.json"
with open(metadata_path) as f:
    metadata = json.load(f)

str2idx = metadata["label"]["str2idx"]
BACKGROUND_CLASS = str2idx["background"]

print("label str2idx:")
for k, v in sorted(str2idx.items(), key=lambda x: x[1]):
    marker = "  <-- background" if k == "background" else ""
    print(f"  {v:3d} : {k!r}{marker}")

print(f"\nBACKGROUND_CLASS = {BACKGROUND_CLASS}")

## Entropic Open-Set loss

The Entropic Open-Set model is trained on the full dataset — known digits plus digits 8–9
labelled as `"background"`.  For known samples the loss is standard cross-entropy.  For
background samples the loss *maximises* Shannon entropy, pushing the softmax output toward
the uniform distribution.

$$
\mathcal{L} =
  \underbrace{-\log p_y}_{\text{CE on known}}
  \;+\;
  \underbrace{\sum_k p_k \log p_k}_{\text{neg-entropy on background}}
$$

In [ ]:
config_entropic = {
    "model_type": "ecd",
    "input_features": [{"name": "image_path", "type": "image", "encoder": ENCODER}],
    "output_features": [
        {
            "name": "label",
            "type": "category",
            "loss": {
                "type": "entropic_open_set",
                "background_class": BACKGROUND_CLASS,
            },
        }
    ],
    "trainer": {"epochs": 10, "learning_rate": 0.001, "batch_size": 128},
}

print(f"background_class = {BACKGROUND_CLASS}")
print(yaml.dump(config_entropic, default_flow_style=False))

model_entropic = LudwigModel(config=config_entropic, logging_level="WARNING")
_, _, output_dir_entropic = model_entropic.train(
    dataset=train_df,
    experiment_name="open_set_mnist",
    model_name="entropic",
    skip_save_processed_input=True,
)
print(f"Entropic model saved to: {output_dir_entropic}")

## Objectosphere loss

The Objectosphere loss extends the Entropic Open-Set loss with a logit-norm objective:

- **Known samples**: CE + hinge `max(0, ξ – ||z||)²` pushes logit L2 norms above ξ
- **Background samples**: entropy maximisation + `ζ||z||²` suppresses logit norms toward zero

This creates a clear norm gap between known and unknown samples, enabling a simple threshold
detector at inference time.

$$
\mathcal{L} =
  \underbrace{\text{CE}(z_{\text{known}}) + \max(0,\,\xi - \|z\|)^2}_{\text{known}}
  \;+\;
  \underbrace{\sum_k p_k \log p_k + \zeta\,\|z\|^2}_{\text{background}}
$$

In [ ]:
config_objectosphere = {
    "model_type": "ecd",
    "input_features": [{"name": "image_path", "type": "image", "encoder": ENCODER}],
    "output_features": [
        {
            "name": "label",
            "type": "category",
            "loss": {
                "type": "objectosphere",
                "background_class": BACKGROUND_CLASS,
                "xi": 10.0,
                "zeta": 0.1,
            },
        }
    ],
    "trainer": {"epochs": 10, "learning_rate": 0.001, "batch_size": 128},
}

model_objectosphere = LudwigModel(config=config_objectosphere, logging_level="WARNING")
_, _, output_dir_objectosphere = model_objectosphere.train(
    dataset=train_df,
    experiment_name="open_set_mnist",
    model_name="objectosphere",
    skip_save_processed_input=True,
)
print(f"Objectosphere model saved to: {output_dir_objectosphere}")

## Compare: confidence on known vs unknown

We now predict on the test set with each model and collect the **maximum softmax probability**
for each sample.  A well-calibrated open-set model should:

- Produce **high** max-prob on known digits (it is confident)
- Produce **low** max-prob on unknown digits (8 and 9) — ideally near `1/num_classes`

In [ ]:
import numpy as np

# Separate test rows into known and unknown
test_known_df   = test_df[~test_df["label"].isin(["8", "9"])].copy()
test_unknown_df = test_df[ test_df["label"].isin(["8", "9"])].copy()

print(f"Test known:   {len(test_known_df)} samples")
print(f"Test unknown: {len(test_unknown_df)} samples")

def get_max_probs(model, df):
    """Return an array of max softmax probabilities for each row in df."""
    preds, _ = model.predict(dataset=df, skip_save_predictions=True)
    return preds["label_probability"].values

print("Predicting with baseline ...")
probs_baseline_known   = get_max_probs(model_baseline,   test_known_df)
probs_baseline_unknown = get_max_probs(model_baseline,   test_unknown_df)

print("Predicting with entropic ...")
probs_entropic_known   = get_max_probs(model_entropic,   test_known_df)
probs_entropic_unknown = get_max_probs(model_entropic,   test_unknown_df)

print("Predicting with objectosphere ...")
probs_obj_known   = get_max_probs(model_objectosphere, test_known_df)
probs_obj_unknown = get_max_probs(model_objectosphere, test_unknown_df)

print("Done.")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
bins = np.linspace(0, 1, 30)

model_data = [
    ("CE Baseline",      probs_baseline_known, probs_baseline_unknown),
    ("Entropic Open-Set", probs_entropic_known, probs_entropic_unknown),
    ("Objectosphere",    probs_obj_known,       probs_obj_unknown),
]

for ax, (title, known, unknown) in zip(axes, model_data):
    ax.hist(known,   bins=bins, alpha=0.6, color="steelblue",  label=f"Known (0-7)\nmean={known.mean():.3f}")
    ax.hist(unknown, bins=bins, alpha=0.6, color="orangered",  label=f"Unknown (8-9)\nmean={unknown.mean():.3f}")
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Max softmax probability")
    ax.legend(fontsize=9)

axes[0].set_ylabel("Count")
fig.suptitle("Confidence on known vs unknown MNIST digits", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("confidence_histograms.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved confidence_histograms.png")

## Threshold-based detection

We can turn max softmax probability into a binary known/unknown detector by choosing a threshold
on a held-out validation set.  A sample with max-prob below the threshold is flagged as unknown.

Below we sweep thresholds and plot the True Positive Rate (TPR — fraction of unknowns correctly
flagged) against the False Positive Rate (FPR — fraction of known samples incorrectly flagged).

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

model_data_thresh = [
    ("CE Baseline",      probs_baseline_known, probs_baseline_unknown),
    ("Entropic Open-Set", probs_entropic_known, probs_entropic_unknown),
    ("Objectosphere",    probs_obj_known,       probs_obj_unknown),
]

for ax, (title, known, unknown) in zip(axes, model_data_thresh):
    # Label: 0 = known, 1 = unknown; detector score = 1 - max_prob
    y_true  = np.concatenate([np.zeros(len(known)), np.ones(len(unknown))])
    y_score = np.concatenate([1 - known, 1 - unknown])

    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)

    ax.plot(fpr, tpr, lw=2, label=f"AUC = {auc:.3f}")
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("False Positive Rate")
    ax.legend(fontsize=10)

axes[0].set_ylabel("True Positive Rate")
fig.suptitle("ROC curve for unknown detection (1 - max_prob threshold)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved roc_curves.png")

## Summary table

The table below summarises the mean maximum softmax probability on known vs unknown test samples.
A lower mean max-prob on unknowns indicates better open-set recognition.

In [ ]:
from sklearn.metrics import roc_auc_score

rows = [
    ("CE Baseline",      probs_baseline_known, probs_baseline_unknown),
    ("Entropic Open-Set", probs_entropic_known, probs_entropic_unknown),
    ("Objectosphere",    probs_obj_known,       probs_obj_unknown),
]

header = f"{'Model':<22} | {'Mean max-prob (known)':>21} | {'Mean max-prob (unknown)':>23} | {'AUC (unknown det.)':>18}"
sep    = "-" * len(header)
print(sep)
print(header)
print(sep)

for name, known, unknown in rows:
    y_true  = np.concatenate([np.zeros(len(known)), np.ones(len(unknown))])
    y_score = np.concatenate([1 - known, 1 - unknown])
    auc     = roc_auc_score(y_true, y_score)
    print(f"{name:<22} | {known.mean():>21.4f} | {unknown.mean():>23.4f} | {auc:>18.4f}")

print(sep)
print()
print("Interpretation:")
print("  - CE Baseline:         high confidence on unknowns — the model cannot say 'I don't know'")
print("  - Entropic Open-Set:   lower mean max-prob on unknowns — closer to uniform distribution")
print("  - Objectosphere:       similar to entropic; also creates a logit-norm gap (not shown here)")